# Unit 3, Lecture 4: Memory, state, and retrieval

Two fixes for two problems. Your agent **forgets** between turns, and it
**invents** facts it was never given.

- **Memory**: a session carries the conversation across turns. This is your
  Unit 2 `Session`, now owned by the framework.
- **Retrieval (RAG)**: search your own documents, put the facts in the prompt,
  and the model answers from them instead of guessing.

The retrieval mechanics (chunk, embed, cosine search) are ordinary code and run
**offline**. Only the real embedding and the final answer need a lane.

## 1. Memory: a session, given not built

The framework gives an agent a session. Pass it to each `run` and the
conversation carries. **This cell needs a lane.**

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from cse476.agent_fw import make_client, build_support_agent

agent = build_support_agent(make_client())
session = agent.create_session()

print(await agent.run("Route ticket 12 to the billing queue.", session=session))
print(await agent.run("Actually, make it urgent.", session=session))
# the second call knows "it" means the billing ticket, because they share a session

That is your Unit 2 `Session` with the manual threading removed. You create
it, pass it to each run, and the history carries. Because you built the plumbing
by hand once, you know exactly what the framework is doing for you.

Memory was hard to build and is easy to use. The rest of this notebook is
retrieval, where the real thinking lives.

## 2. The inventing problem

Ask a model about facts it never saw in training and it produces a fluent,
confident, **wrong** answer. It cannot tell knowing from guessing, so you cannot
prompt it away. The fix is to give it the facts and forbid outside knowledge.

That is retrieval augmented generation. Build it from three parts.

## 3. Chunking

Split documents into pieces small enough to retrieve precisely. Embedding a
whole document as one vector matches it as a blurry average; chunks let you find
the exact paragraph.

In [ ]:
from cse476.rag import chunk_text

doc = ("The billing team handles refunds and invoice disputes. Billing SLA is "
       "24 hours. The technical team handles outages and software bugs. "
       "Technical SLA is 4 hours. The abuse team handles security reports "
       "and account takeovers. Abuse SLA is 1 hour and always escalates.")

for i, c in enumerate(chunk_text(doc, max_words=12)):
    print(f"chunk {i}: {c}")

## 4. Embedding and cosine similarity

An embedding turns text into a vector so that similar meaning becomes geometric
closeness. Cosine similarity is one number measuring that closeness, and it is
the whole engine of search. No model runs here; this is arithmetic.

In [ ]:
from cse476.rag import HashingEmbedder, cosine_similarity

e = HashingEmbedder()
a = e.embed("billing refunds invoice")
b = e.embed("billing refunds invoice")
c = e.embed("technical outages bugs")

print(f"identical texts:  {cosine_similarity(a, b):.3f}")
print(f"different texts:   {cosine_similarity(a, c):.3f}")

## 5. The retrieval store

Put it together: load documents, and search returns the chunks closest in
meaning to a query. Still no model, all offline.

In [ ]:
from cse476.rag import RagStore

store = RagStore(embedder=HashingEmbedder())
store.add_document(doc, source="support-policy.md", max_words=12)

for chunk, score in store.search("billing refunds invoice", top_k=2):
    print(f"{score:.3f}  [{chunk.source}]  {chunk.text}")

## 6. The full RAG shape: retrieve, then generate

Now hand the retrieved context to the agent, with a strict instruction to answer
only from it. **This cell needs a lane.**

In [ ]:
from cse476.rag import RagStore, HashingEmbedder, RAG_INSTRUCTIONS, answer_with_context

# a RAG agent: same framework agent, but instructed to answer only from context
rag_agent = make_client().as_agent(name="policy_rag", instructions=RAG_INSTRUCTIONS)

kb = RagStore(embedder=HashingEmbedder())
kb.add_document(doc, source="support-policy.md", max_words=12)

answer = await answer_with_context(rag_agent, kb, "What is the SLA for billing?")
print(answer)

## 7. The does-not-know test

The most valuable behaviour: when the answer is not in the documents, a good RAG
agent **says so** instead of inventing. Test it deliberately.

In [ ]:
answer = await answer_with_context(rag_agent, kb, "What is the SLA for the legal team?")
print(answer)
# 'legal' is not in the documents. A trustworthy agent admits it does not know.

## 8. The honest limit of the toy embedder

Our teaching embedder matches on shared **words**, not meaning. So it misses
synonyms. This is exactly why real systems use semantic embeddings.

In [ ]:
# "money back" means "refund" but shares no words with the billing chunk
for q in ["refunds", "money back"]:
    hits = store.search(q, top_k=1)
    top = hits[0] if hits else None
    print(f"{q!r:12} -> {top[1]:.3f}  {top[0].text[:45] if top else 'nothing'}")

print()
print("'refunds' matches (shared word); 'money back' does not (no shared word).")
print("A real embedding model would place them near each other by MEANING.")

To use real semantic embeddings, swap `HashingEmbedder` for one built on the
Agent Framework embedding client. Nothing else in `RagStore` changes, because the
embedder sits behind a seam, the same trick as the Transport in Unit 2 Lecture 3:

```python
from agent_framework.openai import OpenAIEmbeddingClient
client = OpenAIEmbeddingClient(model="text-embedding-3-small", api_key=..., base_url=...)
vectors = await client.get_embeddings([text])
```

## Your turn

**1. Build a retrieval store.** Load three or four short documents of your own.
Ask questions and watch which chunks come back. Find one question it retrieves
well and one it retrieves badly.

**2. Break it on purpose.** Ask something that is *not* in any document. Confirm
your RAG agent says it does not know rather than inventing. If it invents,
tighten `RAG_INSTRUCTIONS` until it stops.

**3. Word versus meaning.** Find two phrases that mean the same thing but share
no words. Confirm the toy embedder misses the link, and write one sentence on why
a real embedding would catch it.

In [ ]:
# your work here
